# Lecture 1: First Forward Problem

**Forward problem:** given the state of a physical system now, predict its state a moment later,

$$u(t) \;\longmapsto\; u(t + \Delta t),$$

then apply the model repeatedly to predict further into the future (an *autoregressive rollout*).

1. **Heat equation**: a small dataset we build from the exact solution.
2. **BubbleML**: simulations of pool boiling.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, TensorDataset, ConcatDataset, DataLoader

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

torch.manual_seed(0)
rng = np.random.default_rng(0)

---
## Part 1: The 2D Heat Equation

$$\frac{\partial u}{\partial t} = \alpha \nabla^2 u \quad \text{on } [0,1]^2, \qquad u = 0 \text{ on the boundary}$$

By separation of variables, the solution is a sum of sine modes, each decaying at its own rate:

$$u(x, y, t) = \sum_{m,n} a_{mn} \, \sin(m\pi x)\,\sin(n\pi y)\, e^{-\alpha \pi^2 (m^2 + n^2)\, t}$$

Higher-frequency modes decay faster. We pick random coefficients $a_{mn}$ to get many different
initial conditions, and evaluate this formula on a grid to get our dataset. No solver needed.

In [ ]:
N = 48            # grid points per side
alpha = 1.0       # diffusivity
dt = 5e-4         # time between saved frames
T = 100           # frames per trajectory
K = 6             # use modes m, n = 1..K

x = np.linspace(0, 1, N)
X, Y = np.meshgrid(x, x)

def heat_trajectory():
    u = np.zeros((T + 1, N, N))
    t = np.arange(T + 1)[:, None, None] * dt
    for m in range(1, K + 1):
        for n in range(1, K + 1):
            a = rng.normal() / (m**2 + n**2)     # smaller amplitude for higher frequencies
            u += a * np.sin(m * np.pi * X) * np.sin(n * np.pi * Y) * np.exp(-alpha * np.pi**2 * (m**2 + n**2) * t)
    return u

heat_train = np.stack([heat_trajectory() for _ in range(100)]).astype(np.float32)
heat_test = np.stack([heat_trajectory() for _ in range(20)]).astype(np.float32)
print("dataset shape (trajectory, time, y, x):", heat_train.shape)

In [ ]:
steps = [0, 10, 30, 100]
vmax = np.abs(heat_train[0, 0]).max()
fig, axes = plt.subplots(1, len(steps), figsize=(13, 3), layout="constrained")
for ax, s in zip(axes, steps):
    im = ax.imshow(heat_train[0, s], cmap="RdBu_r", vmin=-vmax, vmax=vmax, origin="lower")
    ax.set_title(f"step {s}"); ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes, label="u  (red = hot, blue = cold)")
fig.suptitle("One heat-equation trajectory: fine structure diffuses away first")
plt.show()

### Setting up the forward problem

Each consecutive pair of frames $(u_n, u_{n+1})$ is one training example: input $u_n$, target $u_{n+1}$.

In [ ]:
def to_pairs(trajs):
    inputs = trajs[:, :-1].reshape(-1, 1, N, N)    # (samples, channels, y, x)
    targets = trajs[:, 1:].reshape(-1, 1, N, N)
    return TensorDataset(torch.from_numpy(inputs), torch.from_numpy(targets))

heat_loader = DataLoader(to_pairs(heat_train), batch_size=64, shuffle=True)
print("training pairs:", len(heat_loader.dataset))

### Model: a small CNN

Convolutions are *local* and *translation invariant*, like the physics of diffusion.
The network predicts the **change** from $u_n$ to $u_{n+1}$, which we add back to the input.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, in_channels=1, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, hidden, 3, padding=1), nn.GELU(),
            nn.Conv2d(hidden, hidden, 3, padding=1), nn.GELU(),
            nn.Conv2d(hidden, hidden, 3, padding=1), nn.GELU(),
            nn.Conv2d(hidden, 1, 3, padding=1),
        )

    def forward(self, x):
        # x[:, :1] is the current field (the first input channel); predict its change
        return x[:, :1] + self.net(x)

def train(model, loader, epochs, lr=1e-3):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        total = 0.0
        for inputs, targets in loader:
            inputs, targets = inputs.float().to(device), targets.float().to(device)   # MPS needs float32
            loss = F.mse_loss(model(inputs), targets)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total += loss.item()
        print(f"epoch {epoch + 1:2d}   loss {total / len(loader):.2e}")
    return model

heat_model = train(SmallCNN(in_channels=1), heat_loader, epochs=10)

### Rollout: using the model as a simulator

Start from the first frame of a test trajectory and feed each prediction back in as the next input.
Each step adds a little error, and those errors build up.

In [ ]:
def relative_error(pred, true):
    return np.linalg.norm(pred - true) / np.linalg.norm(true)

heat_model.eval()
errors = []
for traj in heat_test:
    u = torch.from_numpy(traj[0]).reshape(1, 1, N, N).to(device)
    preds = []
    with torch.no_grad():
        for n in range(T):
            u = heat_model(u)
            preds.append(u.cpu().numpy()[0, 0])
    errors.append([relative_error(preds[n], traj[n + 1]) for n in range(T)])
heat_rollout = np.array(preds)       # rollout of the last test trajectory, kept for plotting

plt.figure(figsize=(5, 3.5))
plt.plot(np.arange(1, T + 1), np.mean(errors, axis=0))
plt.xlabel("rollout step"); plt.ylabel("relative L2 error"); plt.title("Heat equation: rollout error")
plt.show()

In [ ]:
truth = heat_test[-1]
steps = [1, 20, 50, 100]
vmax = np.abs(truth[0]).max()
error = heat_rollout - truth[1:]      # CNN rollout minus ground truth
emax = np.abs(error).max()           # the error is much smaller than the field, so it gets its own color scale
rows = [(truth[1:], "ground truth", "RdBu_r", vmax, "u"),
        (heat_rollout, "CNN rollout", "RdBu_r", vmax, "u"),
        (error, "error", "PuOr_r", emax, "rollout − truth")]
fig, axes = plt.subplots(3, len(steps), figsize=(13, 9), layout="constrained")
for row, (data, name, cmap, lim, label) in enumerate(rows):
    for k, s in enumerate(steps):
        im = axes[row, k].imshow(data[s - 1], cmap=cmap, vmin=-lim, vmax=lim, origin="lower")
        axes[row, k].set_xticks([]); axes[row, k].set_yticks([])
        axes[0, k].set_title(f"step {s}")
    axes[row, 0].set_ylabel(name)
    fig.colorbar(im, ax=axes[row, :], label=label)
plt.show()

---
## Part 2: BubbleML Pool Boiling

[BubbleML](https://github.com/HPCForge/BubbleML) is a dataset of multiphase boiling simulations.
A heater on the bottom wall boils the liquid; vapor bubbles form, grow, detach and rise.

We use the small downsampled *subcooled pool boiling* examples from the BubbleML repo:
three simulations with heater (wall) temperatures of 100, 103 and 106 °C.

In [ ]:
import os
import urllib.request
import h5py

DATA_DIR = "../data/bubbleml"
FILES = ["Twall-100.hdf5", "Twall-103.hdf5", "Twall-106.hdf5"]
URL = "https://raw.githubusercontent.com/HPCForge/BubbleML/main/examples/"

os.makedirs(DATA_DIR, exist_ok=True)
for f in FILES:
    path = os.path.join(DATA_DIR, f)
    if not os.path.exists(path):
        print("downloading", f)
        urllib.request.urlretrieve(URL + f, path)

### What's in a simulation file?

In [ ]:
sim = h5py.File(os.path.join(DATA_DIR, "Twall-100.hdf5"), "r")

for key in ["temperature", "velx", "vely", "pressure", "dfun"]:
    field = sim[key][:]
    print(f"{key:12s} shape {field.shape}   dtype {field.dtype}   range [{field.min():7.3f}, {field.max():7.3f}]")

params = {k.decode().strip(): v for k, v in sim["real-runtime-params"][:]}
print("\nReynolds number:", 1 / params["ins_invreynolds"])
print("domain: x in", (params["xmin"], params["xmax"]), " y in", (params["ymin"], params["ymax"]))

**How is this different from an image or a sequence of tokens?**

* **Channels are physical fields** such as temperature, two velocity components, pressure,
  and `dfun` (signed distance to the bubble surface: > 0 in vapor, < 0 in liquid).
  They have different units and very different ranges.
* **Values are continuous physical quantities** (64 bit floats), not pixel intensities 0–255 or token IDs from a vocabulary.
* **There's a time axis.** Frames are ordered and strongly correlated, like tokens in a sentence, but each
  "token" is an entire 2D field. Shuffling frames destroys the physics.
* **The grid is a physical domain** with real extents and boundary conditions (the heater is the bottom wall).
* **Metadata matters.** Each file records simulation parameters such as the Reynolds number.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# temperature colormap from https://github.com/therml-ai/boiling-viz (for nondimensional temperature in [0, 1])
temp_cmap = LinearSegmentedColormap.from_list("temp_green", list(zip(
    [0.0, 0.02, 0.04, 0.06, 0.08, 0.1, 0.134, 0.167, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    ["#0000FF", "#0443FF", "#0E7AFF", "#16B4FF", "#1FF1FF", "#21FFD3", "#22FF9B", "#22FF67", "#22FF15", "#29FF06",
     "#45FF07", "#6DFF08", "#9EFF09", "#D4FF0A", "#FEF30A", "#FEB709", "#FD7D08", "#FC4908", "#FC1407", "#FB0007"])))

temp = sim["temperature"][:]
speed = np.sqrt(sim["velx"][:]**2 + sim["vely"][:]**2)
dfun = sim["dfun"][:]

t = 50
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
panels = [(temp[t], "temperature", temp_cmap, 0, 1),
          (speed[t], "velocity magnitude", "magma", 0, np.percentile(speed, 99)),
          (dfun[t], "signed distance (dfun)", "RdYlBu", None, None)]
for ax, (img, title, cmap, vmin, vmax) in zip(axes, panels):
    im = ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax, origin="lower")
    ax.contour(dfun[t] > 0, levels=[0.5], colors="black", linewidths=1)   # bubble surface
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=ax, shrink=0.8)
plt.suptitle(f"Twall = 100 °C, timestep {t} (black line = bubble surface)")
plt.tight_layout(); plt.show()

In [ ]:
def animate(fields, titles, filename, cmap=temp_cmap, fps=10):
    # side-by-side animation of 2D fields over time, saved as a GIF; each field has shape (time, y, x)
    fig, axes = plt.subplots(1, len(fields), figsize=(4 * len(fields) + 1, 4), layout="constrained")
    axes = np.atleast_1d(axes)
    ims = []
    for ax, f, title in zip(axes, fields, titles):
        ims.append(ax.imshow(f[0], cmap=cmap, vmin=0, vmax=1, origin="lower"))
        ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(ims[0], ax=axes, label="temperature (nondimensional)")
    label = fig.suptitle("step 0")
    def update(i):
        for im, f in zip(ims, fields):
            im.set_data(f[i])
        label.set_text(f"step {i}")
        return ims
    anim = FuncAnimation(fig, update, frames=len(fields[0]))
    anim.save(filename, writer=PillowWriter(fps=fps), dpi=100)
    plt.close(fig)
    return Image(filename=filename)

animate([temp], ["temperature, Twall = 100 °C"], "bubbleml_temperature.gif")

### The forward problem for boiling

Given **temperature and velocity** at step $n$, predict **temperature** at step $n+1$.

$$(T_n,\; v_{x,n},\; v_{y,n}) \;\longmapsto\; T_{n+1}$$

We train on the 100 °C and 106 °C simulations and test on 103 °C, a condition the model never sees.

In [ ]:
class HDF5Dataset(Dataset):
    def __init__(self, filename):
        self.data = h5py.File(filename, "r")
        self.timesteps = self.data["temperature"].shape[0]

    def __len__(self):
        return self.timesteps - 1

    def __getitem__(self, idx):
        # input: temperature, x-velocity, y-velocity at step idx.  target: temperature at step idx + 1
        input = torch.stack([torch.from_numpy(self.data[k][idx]) for k in ["temperature", "velx", "vely"]])
        label = torch.from_numpy(self.data["temperature"][idx + 1]).unsqueeze(0)
        return input, label

train_files = ["Twall-100.hdf5", "Twall-106.hdf5"]
train_dataset = ConcatDataset([HDF5Dataset(os.path.join(DATA_DIR, f)) for f in train_files])
test_dataset = HDF5Dataset(os.path.join(DATA_DIR, "Twall-103.hdf5"))

bubble_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
print("training pairs:", len(train_dataset), "  test pairs:", len(test_dataset))

In [ ]:
bubble_model = train(SmallCNN(in_channels=3, hidden=64), bubble_loader, epochs=40)

### Rollout on the unseen 103 °C simulation

We only predict temperature, so at every step we take the velocity from the simulation
and the temperature from the model's previous prediction.

In [ ]:
test = test_dataset.data
true_temp = test["temperature"][:].astype(np.float32)
vel = np.stack([test["velx"][:], test["vely"][:]], axis=1).astype(np.float32)   # (time, 2, y, x)

start, steps = 20, 80
bubble_model.eval()
temp_pred = torch.from_numpy(true_temp[start]).reshape(1, 1, 48, 48).to(device)
preds = []
with torch.no_grad():
    for n in range(start, start + steps):
        velocity = torch.from_numpy(vel[n]).unsqueeze(0).to(device)
        temp_pred = bubble_model(torch.cat([temp_pred, velocity], dim=1))
        preds.append(temp_pred.cpu().numpy()[0, 0])
preds = np.array(preds)
truth = true_temp[start + 1 : start + 1 + steps]

plt.figure(figsize=(5, 3.5))
plt.plot(np.arange(1, steps + 1), [relative_error(p, t) for p, t in zip(preds, truth)])
plt.xlabel("rollout step"); plt.ylabel("relative L2 error"); plt.title("BubbleML (103 °C): rollout error")
plt.show()

In [ ]:
animate([truth, preds], ["ground truth", "CNN rollout"], "bubbleml_rollout.gif")

**What to notice:** the error grows much faster than for the heat equation, and the prediction smears out the sharp
bubble boundaries. Keeping long autoregressive rollouts accurate and stable is an open problem.

---
## Exercises

1. **Add the bubble surface as an input.** Include `dfun` as a fourth input channel in `HDF5Dataset`
   (and in the rollout). Does the rollout improve? 
2. **Bigger time steps.** Train the heat equation model to predict 5 frames ahead ($u_n \to u_{n+5}$). Compare its rollout
   error at the same physical time with the 1-step model's. Fewer steps means fewer chances for error to build up but is each step harder?
3. **Unseen frequencies.** Train the heat model with `K = 3` and test on data made with `K = 6`. What goes wrong, and why?
4. **Frozen velocity.** In the BubbleML rollout, keep the velocity fixed at its value at `start` instead of reading it from
   the simulation. How quickly does the prediction fall apart?